# External Data 

### ในส่วนนี้ เราจะทำการดึง *ข้อมูลภายนอก* จำนวน 2 แหล่งหลัก: ###
### 1. ระดับความแออัดของการจราจร (Bangkok Traffic Congestion Level)
- ดึงข้อมูลความแออัดในแต่ละวันและแต่ละช่วงเวลา
- วิธีการดึงข้อมูล:
  1. ใช้ *Selenium* เพื่อโหลดหน้าเว็บแบบ dynamic
  2. ใช้ *BeautifulSoup* เพื่อดึงข้อมูลจาก HTML
- จุดประสงค์:
  - ใช้เป็นตัวอย่างการทำ *web scraping*

### 2. ข้อมูลสภาพอากาศรายชั่วโมงของกรุงเทพ (Bangkok Hourly Weather)
- ใช้เป็นบริบทด้านเวลา (*temporal context*) สำหรับโมเดล ML
- แหล่งข้อมูล:
  - *Open-Meteo API*
- จุดประสงค์:
  1. ดึงข้อมูล เช่น อุณหภูมิ ความชื้น และแรงลม
  2. ใช้ข้อมูลเป็น feature สำหรับ machine learning


In [12]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import pandas as pd
import time

In [13]:
import os
import logging
from typing import List, Dict

import requests

BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_DIR = os.path.join(BASE_DIR, "data")
EXTERNAL_DIR = os.path.join(DATA_DIR, "external")

os.makedirs(EXTERNAL_DIR, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("external_scraping")

In [14]:
df_clean = pd.read_csv('data/bangkok_traffy_clean.csv')

Part 1 – Scraping Bangkok Congestion (BeautifulSoup + Selenium)
แหล่งข้อมูล: TomTom 

โค้ดนี้ใช้ Selenium เพื่อเปิดหน้าเว็บและกดปุ่มโหลดกราฟ จากนั้นใช้ BeautifulSoup ดึงข้อมูลมาจัดเรียงลงตาราง

- เปิดและเข้าถึง (Setup): สั่ง Selenium เปิด Chrome และเข้าหน้าเว็บ TomTom
- คลิกเรียกข้อมูล (Interact): สั่งคลิกแท็บ "Congestion level" (เพราะข้อมูลซ่อนอยู่)
- รอโหลด (Wait): หยุดรอจนกว่าตัวเลข % จะปรากฏขึ้นมาบนหน้าจอ
- ดึงและจัดเรียง (Extract & Parse):
  - ดูด HTML ออกมาแล้วปิด Browser
  - ใช้ BeautifulSoup ตัดคำ แยก "วัน" และ "เวลา"
  - วนลูปเก็บข้อมูลลงตาราง Pandas DataFrame

In [15]:
# URL เป้าหมาย: หน้าข้อมูลจราจรของกรุงเทพฯ
TRAFFICVOL_URL = "https://www.tomtom.com/traffic-index/bangkok-traffic/"

# ==========================================
# 1) Start webdriver (เริ่มการทำงานของหุ่นยนต์)
# ==========================================
# ติดตั้งและเรียกใช้ Chrome Driver โดยอัตโนมัติ
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
driver.get(TRAFFICVOL_URL) # เปิดเว็บไซต์

# สร้างตัวรอ (Wait) สูงสุด 20 วินาที สำหรับใช้รอให้ Element ต่างๆ โหลดเสร็จ
wait = WebDriverWait(driver, 20)

# ==========================================
# 2) Navigate to Tab (ค้นหาและเตรียมคลิกแท็บ)
# ==========================================
try:
    # รอจนกว่า List รายการ (ul) จะปรากฏ เพื่อให้มั่นใจว่าโครงสร้างเว็บโหลดมาแล้ว
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "ul")))
    
    # ค้นหา <li> ที่มีข้อความว่า "Congestion level"
    # ใช้ normalize-space เพื่อตัดช่องว่างหน้าหลังที่อาจจะทำให้หาไม่เจอ
    congestion_li = driver.find_element(By.XPATH, "//li[normalize-space(.)='Congestion level']")
except Exception as e:
    print("Could not locate the 'Congestion level' tab element:", e)
    driver.quit() # ปิด Browser หากหาปุ่มไม่เจอ
    raise

# ==========================================
# 3) Click Interaction (คลิกเพื่อโหลดข้อมูล)
# ==========================================
try:
    congestion_li.click() # ลองคลิกแบบปกติ
except Exception:
    # Fallback: ถ้าคลิกปกติไม่ได้ (เช่น มีแบนเนอร์บัง) ให้ใช้ JavaScript สั่งคลิกโดยตรง
    driver.execute_script("arguments[0].scrollIntoView(true); arguments[0].click();", congestion_li)

print("Clicked 'Congestion level' tab")

# ==========================================
# 4) Wait for Data (รอให้กราฟ/ตัวเลขโหลด)
# ==========================================
# ข้อมูลในหน้านี้เป็น Dynamic (AJAX) ต้องรอให้ตัวเลข % ปรากฏขึ้นมาก่อน
try:
    # รอ Element ที่เป็นช่องใส่ตัวเลข (Class นี้อาจเปลี่ยนได้ในอนาคต หากเว็บอัปเดต)
    wait.until(EC.presence_of_element_located((By.CLASS_NAME, "sc-4b509e6-5")))
except Exception:
    # วิธีสำรอง: วนลูปเช็ค Source Code ว่ามีเครื่องหมาย "%" โผล่มาหรือยัง
    timeout = 10
    end = time.time() + timeout
    while time.time() < end:
        if "%" in driver.page_source:
            break
        time.sleep(0.5)

time.sleep(1)  # รอแถมอีก 1 วินาที เพื่อความชัวร์ว่า Render นิ่งแล้ว

# ==========================================
# 5) Extract HTML (ดึง Source Code ออกมา)
# ==========================================
html = driver.page_source # ดึง HTML ทั้งหมด ณ ปัจจุบัน
driver.quit() # หมดหน้าที่ของ Selenium แล้ว ปิด Browser ได้เลย

# เริ่มใช้ BeautifulSoup แปลง HTML
soup = BeautifulSoup(html, "html.parser")

# ==========================================
# 6) Parse Data (แกะข้อมูลจาก HTML)
# ==========================================
# ค้นหา Container ที่เก็บตารางข้อมูล (Class นี้อิงตามโครงสร้างเว็บ ณ วันที่เขียนโค้ด)
table = soup.find("div", class_="sc-4b509e6-1 gSqKXp")

if table is None:
    print("ERROR: Table container not found after clicking 'Congestion level'.")
else:
    # ดึงทุก div ย่อยที่อยู่ในตารางออกมา
    items = table.find_all('div', recursive=False)
    # แปลง HTML Element เป็นข้อความ (Text) ทั้งหมดเก็บใส่ List
    all_texts = [item.get_text(strip=True) for item in items]

    # --- ส่วนที่ 6.1: หาจุดเริ่มต้นของเวลา (Headers) ---
    try:
        # หาตำแหน่งคำว่า "12:00 AM" เพื่อใช้เป็นจุดเริ่ม
        start_index = all_texts.index("12:00 AM")
    except ValueError:
        start_index = 1  # ค่า Default เผื่อหาไม่เจอ

    # ตัดมา 24 ตัว ถัดจากจุดเริ่ม เพื่อใช้เป็นชื่อคอลัมน์ (เวลา)
    hours = all_texts[start_index : start_index + 24]

    # --- ส่วนที่ 6.2: แยกข้อมูลรายวัน (Grid Values) ---
    # ข้อมูลดิบที่เหลือ (หลังจากส่วนเวลา)
    grid_data = all_texts[start_index + 24 :]
    
    days_of_week = ['Sun', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat']
    traffic_data = []
    current_day = None
    current_day_data = []

    # วนลูปเช็คข้อมูลทีละตัว
    for text in grid_data:
        if text in days_of_week:
            # === เจอชื่อวันใหม่ (เช่น Mon) ===
            
            # บันทึกข้อมูลของ "วันก่อนหน้า" (ถ้ามี) ลง List หลัก
            if current_day and len(current_day_data) == 24:
                row = {'Day': current_day}
                row.update(dict(zip(hours, current_day_data))) # จับคู่เวลากับค่า %
                traffic_data.append(row)

            # เริ่มต้นนับข้อมูลของวันใหม่
            current_day = text
            current_day_data = []
        else:
            # === เจอตัวเลข (Data) ===
            # ถ้ามีชื่อวันอยู่แล้ว ให้เก็บข้อมูลตัวเลขเข้า List ของวันนี้
            if current_day:
                current_day_data.append(text)

    # อย่าลืมบันทึกข้อมูลของ "วันสุดท้าย" (เพราะ Loop จบก่อนจะได้บันทึก)
    if current_day and len(current_day_data) == 24:
        row = {'Day': current_day}
        row.update(dict(zip(hours, current_day_data)))
        traffic_data.append(row)

    # ==========================================
    # 7) Create DataFrame (สร้างตารางแสดงผล)
    # ==========================================
    if traffic_data:
        df = pd.DataFrame(traffic_data).set_index("Day")
        print("Success! Data extracted after clicking 'Congestion level':")
        print(df)
    else:
        print("Error: Extracted NOTHING — grid_data may be empty or structure changed.")

2025-12-07 21:53:48,313 | INFO | WDM | ====== WebDriver manager ======
2025-12-07 21:53:49,836 | INFO | WDM | Get LATEST chromedriver version for google-chrome
2025-12-07 21:53:50,049 | INFO | WDM | Get LATEST chromedriver version for google-chrome
2025-12-07 21:53:50,198 | INFO | WDM | Driver [C:\Users\ten\.wdm\drivers\chromedriver\win64\142.0.7444.175\chromedriver-win32/chromedriver.exe] found in cache


Clicked 'Congestion level' tab
Success! Data extracted after clicking 'Congestion level':
    12:00 AM 01:00 AM 02:00 AM 03:00 AM 04:00 AM 05:00 AM 06:00 AM 07:00 AM  \
Day                                                                           
Sun       0%       0%       0%       0%       0%       0%       0%       2%   
Mon       0%       0%       0%       0%       0%       0%      36%      68%   
Tue       0%       0%       0%       0%       0%       1%      37%      72%   
Wed       0%       0%       0%       0%       0%       1%      37%      71%   
Thu       0%       0%       0%       0%       0%       1%      37%      70%   
Fri       0%       0%       0%       0%       0%       0%      34%      64%   
Sat       1%       0%       0%       0%       0%       0%       0%       9%   

    08:00 AM 09:00 AM  ... 02:00 PM 03:00 PM 04:00 PM 05:00 PM 06:00 PM  \
Day                    ...                                                
Sun       8%      14%  ...      40%      43%    

"เปลี่ยนชื่อหัวตาราง (Columns) จากแบบ AM/PM ให้เป็นเลขชั่วโมง 24 นาฬิกา" เพื่อให้ง่ายต่อการนำไปใช้งานต่อ

In [16]:
data = pd.DataFrame(traffic_data)

In [21]:
data.columns

Index(['Day', '12:00 AM', '01:00 AM', '02:00 AM', '03:00 AM', '04:00 AM',
       '05:00 AM', '06:00 AM', '07:00 AM', '08:00 AM', '09:00 AM', '10:00 AM',
       '11:00 AM', '12:00 PM', '01:00 PM', '02:00 PM', '03:00 PM', '04:00 PM',
       '05:00 PM', '06:00 PM', '07:00 PM', '08:00 PM', '09:00 PM', '10:00 PM',
       '11:00 PM'],
      dtype='object')

In [30]:
import pandas as pd

# สมมติว่าโหลดไฟล์แล้ว
congestion_level = pd.read_csv("bangkok_congestion_level.csv")

def convert_to_24h(col):
    # แก้ตรงนี้: เช็ค 'Day' (ตัวใหญ่) หรือใช้ .lower() เพื่อความชัวร์
    if col.lower() == "day":
        return col
    
    # แปลงเวลา
    t = pd.to_datetime(col, format="%I:%M %p")
    return t.strftime("%H")  # เก็บแค่ "00"–"23"

# แก้ตรงนี้: เติม s ที่ .columns
congestion_level.columns = [convert_to_24h(c) for c in congestion_level.columns]

# เช็คผลลัพธ์
print(congestion_level.columns)

Index(['Day', '00', '01', '02', '03', '04', '05', '06', '07', '08', '09', '10',
       '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22',
       '23'],
      dtype='object')


ทำการ กลับด้านตาราง (Unpivot) ข้อมูลจราจรให้เป็นแนวยาว แล้วทำความสะอาดข้อมูล ก่อนจะนำไป merge เข้ากับตารางข้อมูลหลัก

In [31]:
# 1. Melt: กลับด้านข้อมูล
df_congestion = congestion_level.melt(
    id_vars='Day', 
    var_name='hour', 
    value_name='congestion'
)

# 2. แก้ไขตรงนี้: 
# เนื่องจาก header เป็น "00", "01" อยู่แล้ว ไม่ต้องใช้ pd.to_datetime
# ให้แปลงเป็น integer ได้เลย
df_congestion['hour'] = df_congestion['hour'].astype(int)

# 3. Clean Congestion: แปลง "35%" -> 35
df_congestion['congestion'] = df_congestion['congestion'].str.replace('%','').astype(int)

# 4. Merge
df_merged = df_clean.merge(
    df_congestion, 
    left_on=['day','hour'], 
    right_on=['Day','hour'], 
    how='left'
)
df_merged = pd.DataFrame(df_merged)
df_merged.head()

,ticket_id,type,organization,comment,district,timestamp,star,count_reopen,day,hour,month,year,lat,lon,resolution_time_hours,isLate,urgency_score,Day,congestion
0,2021-FYJTFP,{ความสะอาด},เขตบางซื่อ,ขยะเยอะ,บางซื่อ,2021-09-03 12:51:09.453003+00:00,0.0,0,Fri,12,9,2021,100.53084,13.81865,6578.718099,1,2,Fri,58
1,2021-CGPMUN,"{น้ำท่วม,ร้องเรียน}","เขตประเวศ,ฝ่ายโยธา เขตประเวศ",น้ำท่วมเวลาฝนตกและทะลุเข้าบ้านเดือดร้อนมากทุกๆ...,ประเวศ,2021-09-19 14:56:08.924992+00:00,4.0,0,Sun,14,9,2021,100.66709,13.67891,6593.416835,1,5,Sun,40
2,2021-7XATFA,{สะพาน},เขตสาทร,สะพานลอยปรับปรุงไม่เสร็จตามกำหนด\nปากซอย สาทร12,สาทร,2021-09-26 05:03:52.594898+00:00,0.0,0,Sun,5,9,2021,100.52649,13.72060,6068.222133,1,5,Sun,0
3,2021-9U2NJT,{น้ำท่วม},"เขตบางซื่อ,ฝ่ายโยธา เขตบางซื่อ",น้ำท่วม,บางซื่อ,2021-10-14 10:45:27.713884+00:00,0.0,0,Thu,10,10,2021,100.53099,13.81853,7893.837797,1,5,Thu,59
4,2021-DVEWYM,"{น้ำท่วม,ถนน}","เขตลาดพร้าว,ฝ่ายโยธา เขตลาดพร้าว",ซอยลาดพร้าววังหิน 75 ถนนลาดพร้าววังหิน แขวงลาด...,ลาดพร้าว,2021-12-09 12:29:08.408763+00:00,5.0,0,Thu,12,12,2021,100.59165,13.82280,5898.826799,1,5,Thu,54


## Part 2: การดึงข้อมูลสภาพอากาศรายชั่วโมง (Open-Meteo API)

### 1. แหล่งข้อมูล (Data Source)
* **Source:** [Open-Meteo Historical Weather API](https://open-meteo.com/)
* **Accessibility:** ฟรี ไม่ต้องใช้ API Key
* **Objective:** ดึงข้อมูลสภาพอากาศย้อนหลังของกรุงเทพฯ เพื่อนำมาหาความสัมพันธ์ (Correlation) กับข้อมูลการแจ้งเหตุ

### 2. รายละเอียดข้อมูล (Data Specification)
เราจะดึงข้อมูลตลอดทั้งปี **2023** เพื่อให้ได้จำนวนข้อมูลที่มากเพียงพอ (> 8,000 แถวต่อชั่วโมง) ซึ่งผ่านเกณฑ์ขั้นต่ำ $\ge 1,000$ แถว

| พารามิเตอร์ | ค่าที่กำหนด / คำอธิบาย |
| :--- | :--- |
| **Location** | กรุงเทพมหานคร (Lat: `13.7563`, Lon: `100.5018`) |
| **Time Range** | `2023-01-01` ถึง `2023-12-31` |
| **Variables** | ปริมาณฝน (`precipitation`), อุณหภูมิ (`temperature_2m`), ความเร็วลม (`wind_speed_10m`) |

### 3. มุมมองด้าน Machine Learning (Feature Engineering)
สภาพอากาศถือเป็นปัจจัยภายนอก (Exogenous Variables) ที่ส่งผลกระทบโดยตรงต่อการปฏิบัติงานภาคสนาม

* **ฟีเจอร์ที่น่าสนใจ (Potential Features):**
    * `is_rainy_hour`: ตัวแปร Binary (เป็น 1 หากฝนตกเกินเกณฑ์ที่กำหนด, ถ้าไม่ตกเป็น 0)
    * `rain_mm_last_3h`: ปริมาณฝนสะสม 3 ชั่วโมงย้อนหลัง (ใช้จับสัญญาณความเสี่ยงน้ำท่วมขัง)
    * `temp_2m`, `wind_speed_10m`: ค่าตัวเลขดิบของอุณหภูมิและลม

* **สมมติฐานทางธุรกิจ (Business Hypotheses):**
    1.  **Delay Factor:** ฝนตกหนักหรือลมแรง $\rightarrow$ การเดินทางและการซ่อมแซมทำได้ยากขึ้น $\rightarrow$ เพิ่มโอกาสที่งานจะเสร็จล่าช้า (**"Late"**)
    2.  **Volume Spike:** สภาพอากาศแย่มักสัมพันธ์กับจำนวนการแจ้งเหตุบางประเภทที่พุ่งสูงขึ้น (เช่น น้ำท่วม, ถนนชำรุด, ต้นไม้ล้ม)

In [32]:
from datetime import date

# พิกัดของกรุงเทพโดยคร่าวๆ
BKK_LAT = 13.75
BKK_LON = 100.50

# กำหนดช่วงเวลาที่จะใช้ (ครอบคลุมช่วงเวลาของ Traffy Fondue)
start_date = "2024-01-01"
end_date = "2024-12-31"

# api ของสภาพอากาศ
WEATHER_URL = "https://archive-api.open-meteo.com/v1/era5"

# param ที่ใช้ขอข้อมูล
params = {
    "latitude": BKK_LAT,
    "longitude": BKK_LON,
    "start_date": start_date,
    "end_date": end_date,
    "hourly": ["temperature_2m", "precipitation", "wind_speed_10m"],
    "timezone": "Asia/Bangkok",
}

# log และขอข้อมูล
logger.info("Requesting hourly weather from Open-Meteo...")
resp = requests.get(WEATHER_URL, params=params, timeout=40)
resp.raise_for_status()
weather_json = resp.json()

# แปลง JSON ให้เป็น DataFrame
hourly = weather_json["hourly"]
df_weather = pd.DataFrame(hourly)

logger.info("Weather rows: %d", len(df_weather))
df_weather.head(5)

2025-12-07 22:04:36,313 | INFO | external_scraping | Requesting hourly weather from Open-Meteo...
2025-12-07 22:04:37,886 | INFO | external_scraping | Weather rows: 8784


,time,temperature_2m,precipitation,wind_speed_10m
0,2024-01-01T00:00,27.2,0.0,10.9
1,2024-01-01T01:00,26.3,0.0,10.5
2,2024-01-01T02:00,25.6,0.0,7.7
3,2024-01-01T03:00,25.0,0.0,5.3
4,2024-01-01T04:00,25.2,0.0,8.0


In [33]:
# Refactor ขื่อ column เวลา
df_weather.rename(columns={"time": "datetime"}, inplace=True)

In [34]:

# weather_path = os.path.join(EXTERNAL_DIR, "bangkok_hourly_weather_2023.csv")
# df_weather.to_csv(weather_path, index=False, encoding="utf-8-sig")

# logger.info("Saved hourly weather data to: %s", weather_path)


import os
out_dir = 'data'
os.makedirs(out_dir, exist_ok=True)
weather_path = os.path.join(out_dir, "bangkok_hourly_weather_2024.csv")
df_weather.to_csv(weather_path, index=False, encoding='utf-8-sig')
print('Saved cleaned traffy to', weather_path)
#-------------------------------------------------------------------------
merge_congestion_path = os.path.join(out_dir, "bangkok_merge_congestion.csv")
df_merged.to_csv(merge_congestion_path, index=False, encoding='utf-8-sig')
print('Saved cleaned traffy to', merge_congestion_path)

Saved cleaned traffy to data\bangkok_hourly_weather_2024.csv
Saved cleaned traffy to data\bangkok_merge_congestion.csv


In [35]:
df_weather.to_csv(weather_path, index=False, encoding="utf-8-sig")